# Thử nghiệm shot captioning: OmniShotCut + VideoLLaMA3 2B

Chạy trên **Google Colab có GPU** và bật Internet. Đặt một hoặc nhiều video `.mp4` trong `/content/videos/`. Mặc định notebook xử lý **1 video đầu tiên** để thử nhỏ; đổi `MAX_VIDEOS` trong cell cấu hình nếu cần.

1. OmniShotCut v1.5 tìm ranh giới shot.
2. Shot dài hơn **10 giây** được tách thành các đoạn ngắn (vẫn giữ ID shot gốc). VideoLLaMA3-2B viết **một caption chi tiết cho mỗi đoạn tối đa 10 giây**. Mốc thời gian trong JSON là khoảng video đã đưa vào model, không phải thời điểm model tự dự đoán cho từng hành động.

Kết quả: `/content/caption_results/<video_id>/shots.json` và `captions.json` để tiếp tục khi chạy lại; `/content/caption_results/<video_id>.json` là danh sách segment cuối cùng với `video_id`, `start`, `end`, `caption` như ASR. Các caption gần trùng của những đoạn liền kề được gộp thời gian. Notebook đọc trực tiếp từng khoảng của video gốc; mô tả không dựa trên âm thanh.

API tham khảo: [OmniShotCut](https://github.com/UVA-Computer-Vision-Lab/OmniShotCut), [VideoLLaMA3](https://github.com/DAMO-NLP-SG/VideoLLaMA3).


In [ ]:
# 1. Cài gói theo hướng dẫn inference của VideoLLaMA3 và OmniShotCut
import shutil, subprocess, sys
assert shutil.which("ffmpeg") and shutil.which("ffprobe"), "Colab cần ffmpeg và ffprobe"
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.46.3", "accelerate==1.0.1",
                "decord", "ffmpeg-python",
                "git+https://github.com/UVA-Computer-Vision-Lab/OmniShotCut.git"], check=True)
print("Đã cài dependency. Nếu Colab yêu cầu restart runtime, restart rồi chạy lại từ đầu.")


In [ ]:
# 2. Cấu hình thử nghiệm
from pathlib import Path
VIDEO_DIR = Path("/content/videos")
OUTPUT_DIR = Path("/content/caption_results")
MAX_VIDEOS = 1                # None để xử lý toàn bộ *.mp4
MAX_SHOT_SECONDS = 10.0       # tách shot OmniShotCut dài hơn ngưỡng này
FRAME_SAMPLE_FPS = 1
CAPTION_PROFILES = [
    {"max_frames": 8, "short_side": 224},
    {"max_frames": 4, "short_side": 192},
    {"max_frames": 2, "short_side": 160},
]  # thử mức kế tiếp nếu GPU hết bộ nhớ
MAX_NEW_TOKENS = 220
CAPTION_LANGUAGE = "English"
MIN_CAPTION_SECONDS = 1.0   # đoạn ngắn hơn không đủ mẫu ở 1 frame/giây
DUPLICATE_MAX_GAP = 0.25  # chỉ xét các caption liền kề
DUPLICATE_SIMILARITY = 0.90
OVERWRITE = False

SHOT_MODEL = "uva-cv-lab/OmniShotCut_v1.5"
CAPTION_MODEL = "DAMO-NLP-SG/VideoLLaMA3-2B"

assert 0 < MIN_CAPTION_SECONDS <= MAX_SHOT_SECONDS <= 10 and FRAME_SAMPLE_FPS > 0
assert 0 <= DUPLICATE_MAX_GAP and 0 < DUPLICATE_SIMILARITY <= 1
assert all(p["max_frames"] > 0 and p["short_side"] > 0 for p in CAPTION_PROFILES)
videos = sorted(p for p in VIDEO_DIR.glob("*.mp4") if p.is_file())
if not videos:
    raise FileNotFoundError(f"Không thấy .mp4 trong {VIDEO_DIR}")
if MAX_VIDEOS is not None:
    videos = videos[:MAX_VIDEOS]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Sẽ xử lý:", [p.name for p in videos])


In [ ]:
# 3. Step 1 — phát hiện shot bằng OmniShotCut
import gc, json, math
from fractions import Fraction
import torch
import omnishotcut

if not torch.cuda.is_available():
    raise RuntimeError("Hãy bật GPU trong Runtime > Change runtime type")
def probe_video(path):
    data = json.loads(subprocess.check_output(
        ["ffprobe", "-v", "error", "-select_streams", "v:0",
         "-show_entries", "stream=avg_frame_rate,duration:format=duration",
         "-of", "json", str(path)], text=True))
    stream = data["streams"][0]
    fps = float(Fraction(stream["avg_frame_rate"]))
    duration = float(data["format"].get("duration") or stream["duration"])
    if fps <= 0 or duration <= 0:
        raise ValueError(f"FPS/duration không hợp lệ: {path}")
    return fps, duration

def save_json(path, payload):
    tmp = path.with_suffix(".json.tmp")
    tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(path)

shot_detector = None  # chi nap neu can phat hien shot moi
shot_files = []
for video in videos:
    video_out = OUTPUT_DIR / video.stem
    video_out.mkdir(parents=True, exist_ok=True)
    shot_path = video_out / "shots.json"
    if shot_path.exists() and not OVERWRITE:
        shots_doc = json.loads(shot_path.read_text(encoding="utf-8"))
        if (shots_doc.get("video_path") != str(video.resolve()) or
            shots_doc.get("video_size") != video.stat().st_size or
            shots_doc.get("video_mtime_ns") != video.stat().st_mtime_ns or
            shots_doc.get("shot_model") != SHOT_MODEL or
            shots_doc.get("max_shot_seconds") != MAX_SHOT_SECONDS):
            raise RuntimeError(f"Kiểm tra file cũ trước khi dùng lại: {shot_path}")
    else:
        fps, duration = probe_video(video)
        if shot_detector is None:
            shot_detector = omnishotcut.load(SHOT_MODEL, filename="OmniShotCut_ckpt.pth")
        ranges = shot_detector.inference(str(video), mode="clean_shot")
        if len(ranges) == 0:
            ranges = [[0, max(1, round(duration * fps))]]
        shots = []
        for source_shot_id, (start_frame, end_frame) in enumerate(ranges, 1):
            start_frame, end_frame = int(start_frame), int(end_frame)
            source_start = max(0.0, min(duration, start_frame / fps))
            source_end = max(0.0, min(duration, end_frame / fps))  # end exclusive
            if source_end <= source_start:
                continue
            parts = math.ceil((source_end - source_start) / MAX_SHOT_SECONDS)
            for part in range(parts):
                start = source_start + (source_end - source_start) * part / parts
                end = source_end if part == parts - 1 else source_start + (source_end - source_start) * (part + 1) / parts
                shots.append({"shot_id": len(shots) + 1,
                              "source_shot_id": source_shot_id,
                              "start": round(start, 3), "end": round(end, 3)})
        if not shots:
            raise RuntimeError(f"OmniShotCut không trả về shot hợp lệ: {video}")
        shots_doc = {"video_id": video.stem, "video_path": str(video.resolve()),
                     "video_size": video.stat().st_size,
                     "video_mtime_ns": video.stat().st_mtime_ns,
                     "fps": fps, "duration": duration, "shot_model": SHOT_MODEL,
                     "max_shot_seconds": MAX_SHOT_SECONDS,
                     "shots": shots}
        save_json(shot_path, shots_doc)
    shot_files.append(shot_path)
    print(f"{video.name}: {len(shots_doc['shots'])} shot -> {shot_path}")

del shot_detector
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# 4. Step 2 — nạp VideoLLaMA3-2B; float16 để chạy được trên T4
from transformers import AutoModelForCausalLM, AutoProcessor

caption_model = AutoModelForCausalLM.from_pretrained(
    CAPTION_MODEL, trust_remote_code=True, device_map={"": "cuda:0"},
    torch_dtype=torch.float16,
)
caption_model.eval()
caption_processor = AutoProcessor.from_pretrained(CAPTION_MODEL, trust_remote_code=True)
print("Đã nạp", CAPTION_MODEL)


In [ ]:
# 5. Caption each shot (<=10 s); keep checkpoints, export ASR-style JSON
from IPython.display import display, Markdown
from decord import VideoReader, cpu
from difflib import SequenceMatcher
import re
import cv2
import numpy as np

settings = {
    "max_shot_seconds": MAX_SHOT_SECONDS,
    "frame_sample_fps": FRAME_SAMPLE_FPS,
    "caption_profiles": CAPTION_PROFILES,
    "max_new_tokens": MAX_NEW_TOKENS,
    "language": CAPTION_LANGUAGE,
    "min_caption_seconds": MIN_CAPTION_SECONDS,
}

def gpu_memory_gib():
    return (torch.cuda.memory_allocated() / 2**30,
            torch.cuda.memory_reserved() / 2**30)

def normalize_caption(text):
    return re.sub(r"[^\w]+", " ", text.casefold()).strip()

def merge_duplicate_captions(video_id, raw_shots):
    segments = []
    for shot in sorted(raw_shots, key=lambda item: (item["start"], item["end"])):
        current = {"video_id": video_id, "start": shot["start"],
                   "end": shot["end"], "caption": shot["caption"].strip()}
        if segments:
            previous = segments[-1]
            gap = current["start"] - previous["end"]
            similarity = SequenceMatcher(None, normalize_caption(previous["caption"]),
                                         normalize_caption(current["caption"])).ratio()
            if -0.001 <= gap <= DUPLICATE_MAX_GAP and similarity >= DUPLICATE_SIMILARITY:
                previous["end"] = max(previous["end"], current["end"])
                continue
        segments.append(current)
    return segments

def sample_interval_frames(reader, start, end, profile):
    fps = reader.get_avg_fps()
    first = max(0, math.ceil(start * fps))
    last = min(len(reader) - 1, math.ceil(end * fps) - 1)  # end exclusive
    if first > last:
        raise ValueError(f"No decoded frames in [{start:.3f}, {end:.3f}] (fps={fps:.3f})")
    count = min(profile["max_frames"], max(1, math.ceil((end - start) * FRAME_SAMPLE_FPS)))
    indices = np.linspace(first, last, min(count, last - first + 1), dtype=int).tolist()
    raw_frames = reader.get_batch(indices).asnumpy()
    frames = []
    for frame in raw_frames:
        height, width = frame.shape[:2]
        scale = profile["short_side"] / min(height, width)
        resized = cv2.resize(frame, (max(1, round(width * scale)),
                                     max(1, round(height * scale))), interpolation=cv2.INTER_AREA)
        frames.append(np.ascontiguousarray(resized.transpose(2, 0, 1)))
    timestamps = [index / fps for index in indices]
    assert frames and len(frames) == len(timestamps)
    return frames, timestamps

def caption_interval(reader, video_path, start, end):
    prompt = (
        f"Describe this short video segment in {CAPTION_LANGUAGE}. "
        "Describe the main visible subject, setting, surroundings, and what the subject does. "
        "Describe the visible actions and changes in chronological order, including "
        "meaningful interactions, camera movement, or timelapse segments when visible. "
        "Write 2-4 concrete sentences. "
        "Only state what can be seen; do not infer speech, sounds, names, or intent. "
        "Do not repeat generic phrases or invent unseen events. "
        "If a detail is absent, omit it rather than saying it is not there."
    )
    for attempt, profile in enumerate(CAPTION_PROFILES, 1):
        inputs, ids, frames = None, None, None
        try:
            frames, timestamps = sample_interval_frames(reader, start, end, profile)
            conversation = [
                {"role": "system", "content": "You are a precise video captioner."},
                {"role": "user", "content": [
                    {"type": "video", "video": frames,
                     "num_frames": len(frames), "timestamps": timestamps},
                    {"type": "text", "text": prompt},
                ]},
            ]
            inputs = caption_processor(
                conversation=conversation, images=[("video", frames)], add_system_prompt=True,
                add_generation_prompt=True, return_tensors="pt")
            inputs = {k: v.to("cuda:0") if isinstance(v, torch.Tensor) else v
                      for k, v in inputs.items()}
            if "pixel_values" in inputs:
                inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)
            with torch.inference_mode():
                ids = caption_model.generate(
                    **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
            # The original VideoLLaMA3 model returns generated ids for inputs_embeds.
            caption = caption_processor.batch_decode(
                ids, skip_special_tokens=True)[0].strip()
            if not caption:
                raise RuntimeError(f"Empty caption: {video_path} [{start}, {end}]")
            return caption, {"frame_sample_fps": FRAME_SAMPLE_FPS, **profile,
                             "sampled_frames": len(frames)}
        except torch.cuda.OutOfMemoryError:
            print(f"GPU OOM [{start:.2f}-{end:.2f}s] with "
                  f"{profile['max_frames']} frames/{profile['short_side']}px")
            if attempt == len(CAPTION_PROFILES):
                raise
        finally:
            del inputs, ids, frames
            gc.collect()
            torch.cuda.empty_cache()

for video, shot_path in zip(videos, shot_files):
    reader = VideoReader(str(video), ctx=cpu(0), num_threads=2)
    shot_doc = json.loads(shot_path.read_text(encoding="utf-8"))
    result_path = shot_path.with_name("captions.json")
    if result_path.exists() and not OVERWRITE:
        result = json.loads(result_path.read_text(encoding="utf-8"))
        if (result.get("video_path") != str(video.resolve()) or
            result.get("model") != CAPTION_MODEL or
            result.get("shots_source") != shot_doc["shots"]):
            raise RuntimeError(f"Video/shot changed; check {result_path}")
        previous_settings = result.get("settings", {})
        if previous_settings != settings:
            # Older completed shots keep the sampling settings actually used.
            for prior in result["shots"]:
                prior.setdefault("sampling", {
                    "frame_sample_fps": previous_settings.get("frame_sample_fps"),
                    "max_frames": previous_settings.get("max_frames_per_description"),
                    "short_side": 336,
                })
            history = result.setdefault("settings_history", [])
            for item in (previous_settings, settings):
                if item and item not in history:
                    history.append(item)
            result["settings"] = settings
            save_json(result_path, result)
    else:
        result = {"video_id": video.stem, "video_path": str(video.resolve()),
                  "model": CAPTION_MODEL, "shot_model": SHOT_MODEL,
                  "settings": settings, "settings_history": [settings],
                  "shots_source": shot_doc["shots"], "shots": []}
    done_ids = {item["shot_id"] for item in result["shots"]}
    for shot in shot_doc["shots"]:
        if shot["shot_id"] in done_ids:
            continue
        start, end = shot["start"], shot["end"]
        if not 0 < end - start <= MAX_SHOT_SECONDS + 0.001:
            raise ValueError(f"Shot exceeds 10 seconds: {shot}")
        if end - start < MIN_CAPTION_SECONDS:
            print(f"Skip very short shot {shot['shot_id']} [{start:.2f}-{end:.2f}s]")
            continue
        gc.collect()
        torch.cuda.empty_cache()
        before_alloc, before_reserved = gpu_memory_gib()
        torch.cuda.reset_peak_memory_stats()
        caption, sampling = caption_interval(reader, video, start, end)
        after_alloc, after_reserved = gpu_memory_gib()
        peak = torch.cuda.max_memory_allocated() / 2**30
        print(f"GPU GiB before allocated/reserved={before_alloc:.2f}/{before_reserved:.2f}; "
              f"after={after_alloc:.2f}/{after_reserved:.2f}; peak allocated={peak:.2f}")
        result["shots"].append({**shot, "caption": caption, "sampling": sampling})
        save_json(result_path, result)
        print(f"{video.name} shot {shot['shot_id']} [{start:.2f}-{end:.2f}s]: {caption}")
    segments = merge_duplicate_captions(video.stem, result["shots"])
    export_path = OUTPUT_DIR / f"{video.stem}.json"
    save_json(export_path, segments)
    print(f"Done {video.name}: {len(result['shots'])} captions -> {len(segments)} segments")
    display(Markdown(f"**Result:** `{export_path}` (checkpoint: `{result_path}`)"))
    del reader
    gc.collect()